In [ ]:
using Pkg
pkg"add Oceananigans, NCDatasets, CairoMakie, ConservativeRegridding"
pkg"add CUDA" # or Metal, or AMDGPU

In [ ]:
using Oceananigans
using Oceananigans.Units
using Oceananigans.Architectures: on_architecture
using Oceananigans.Coriolis: DualGridScheme
using Oceananigans.Grids: φnode
using Oceananigans.ImmersedBoundaries: InterfaceImmersedCondition
using NCDatasets
using Downloads
using Printf
using CairoMakie
using ConservativeRegridding

In [ ]:
arch = if Sys.isapple() && Sys.ARCH === :aarch64
    using Metal
    Metal.functional() ? GPU(Metal.MetalBackend()) : CPU()
elseif !isnothing(Sys.which("nvidia-smi"))
    using CUDA
    CUDA.functional() ? GPU() : CPU()
elseif !isnothing(Sys.which("rocminfo"))
    using AMDGPU
    AMDGPU.functional() ? GPU(AMDGPU.ROCBackend()) : CPU()
else
    CPU()
end

FT = Float32
Oceananigans.defaults.FloatType = FT

In [ ]:
resolution = 1 # degrees
Nx = round(Int, 360 / resolution)
Ny = Nx ÷ 2
z_faces = [-4000, -1500, -500, -100, 0]
Nz = length(z_faces) - 1
z = MutableVerticalDiscretization(z_faces)

underlying_grid = TripolarGrid(arch; size=(Nx, Ny, Nz), z, halo=(5, 5, 5))

In [ ]:
stride = 10 # arc-minutes between samples
url = "https://www.ngdc.noaa.gov/thredds/dodsC/global/ETOPO1_Ice_g_gmt4.nc"

etopo_elevation = NCDataset(url) do dataset
    i = 1 + stride÷2 : stride : dataset.dim["lon"] - stride÷2
    j = 1 + stride÷2 : stride : dataset.dim["lat"] - stride÷2
    nomissing(dataset["z"][i, j])
end

In [ ]:
etopo_grid = LatitudeLongitudeGrid(arch; size = size(elevation),
                                   longitude = (-180, 180),
                                   latitude = (-90, 90),
                                   topology = (Periodic, Bounded, Flat))

elevation_field = CenterField(etopo_grid)
set!(elevation_field, elevation)

bottom_height = Field{Center, Center, Nothing}(underlying_grid)
interpolate!(bottom_height, elevation_field)

grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height, InterfaceImmersedCondition()); active_cells_map=true)

In [ ]:
height = bottom_height_field(grid)
bathymetry = Field{Center, Center, Nothing}(grid)
set!(bathymetry, -height)

longitude_ticks = (120:60:420, ["120°E", "180°", "120°W", "60°W", "0°", "60°E"])

map_axis(fig_position; title="", limits=((70, 430), (-80, 70)), xticks=longitude_ticks) =
    Axis(fig_position; title, limits, xticks, aspect=DataAspect(),
                       xlabel="Longitude", ylabel="Latitude")

fig = Figure(size=(900, 500))
ax = map_axis(fig[1, 1]; title="Ocean depth")
qm = quadmesh!(ax, bathymetry; colormap=:deep, nan_color=:gray)
Colorbar(fig[1, 2], qm, label="Depth [m]")

In [ ]:
wind_belts = (northern_trades = FT(-0.055), southern_trades = FT(-0.065),
              northern_westerlies = FT(0.07), southern_westerlies = FT(0.19)) # [N m⁻²]
ρ₀ = 1020 # reference density [kg m⁻³]

wind_belt(φ, τ, φ₀) = τ * exp(-((φ - φ₀) / 11)^2)

zonal_wind_stress(φ, belts) = wind_belt(φ, belts.northern_trades, 18) + wind_belt(φ, belts.southern_trades, -17) +
                              wind_belt(φ, belts.northern_westerlies, 46) + wind_belt(φ, belts.southern_westerlies, -51)

zonal_momentum_flux(λ, φ, t, parameters) = - zonal_wind_stress(φ, parameters.wind_belts) / parameters.ρ₀

In [ ]:
R = Oceananigans.defaults.planet_radius
Ω = Oceananigans.defaults.planet_rotation_rate

wind_stress_curl(φ) = - (zonal_wind_stress(φ + 0.01, wind_belts) - zonal_wind_stress(φ - 0.01, wind_belts)) / (R * deg2rad(0.02))
sverdrup_transport(φ, rotation_rate) = wind_stress_curl(φ) / (ρ₀ * 2 * rotation_rate * cosd(φ) / R)

latitudes = -80:0.5:80
extratropical_latitudes = filter(φ -> abs(φ) > 5, latitudes)

fig = Figure(size=(800, 400))
ax = Axis(fig[1, 1], xlabel="Zonal wind stress [N m⁻²]", ylabel="Latitude [°]")
lines!(ax, [zonal_wind_stress(φ, wind_belts) for φ in latitudes], latitudes)
ax = Axis(fig[1, 2], xlabel="Sverdrup transport [m² s⁻¹]", ylabel="Latitude [°]")
lines!(ax, sverdrup_transport.(extratropical_latitudes, Ω), extratropical_latitudes)

In [ ]:
wind_stress = FluxBoundaryCondition(zonal_momentum_flux, parameters=(; wind_belts, ρ₀))
drag = BulkDrag(coefficient=FT(2.5e-3))
u_boundary_conditions = FieldBoundaryConditions(top=wind_stress, bottom=drag, immersed=ImmersedBoundaryCondition(bottom=drag))
v_boundary_conditions = FieldBoundaryConditions(bottom=drag, immersed=ImmersedBoundaryCondition(bottom=drag))

In [ ]:
restoring_temperature(φ) = 30 * cos(deg2rad(φ))^2

@inline function temperature_flux(i, j, grid, clock, fields, parameters)
    φ = φnode(i, j, grid.Nz, grid, Center(), Center(), Center())
    return @inbounds parameters.rate * (fields.T[i, j, grid.Nz] - restoring_temperature(φ))
end

restoring_rate = FT(100 / 30days) # surface layer thickness over the restoring time scale [m s⁻¹]
temperature_restoring = FluxBoundaryCondition(temperature_flux; discrete_form=true, parameters=(; rate=restoring_rate))
T_boundary_conditions = FieldBoundaryConditions(top=temperature_restoring)

In [ ]:
Δt = 1.5hour
free_surface = SplitExplicitFreeSurface(grid; cfl=0.7, fixed_Δt=Δt)

In [ ]:
momentum_advection = WENOVectorInvariant(order=5)
tracer_advection = WENO(order=7)
buoyancy = SeawaterBuoyancy(equation_of_state=LinearEquationOfState(thermal_expansion=2e-4), constant_salinity=35)

In [ ]:
closure = ConvectiveAdjustmentVerticalDiffusivity(convective_κz=1, convective_νz=1)

function build_model(grid, coriolis)
    model = HydrostaticFreeSurfaceModel(grid; coriolis, free_surface, buoyancy, closure,
                                        momentum_advection, tracer_advection,
                                        tracers = :T,
                                        timestepper = :SplitRungeKutta3,
                                        vertical_coordinate = ZStarCoordinate(),
                                        boundary_conditions = (u=u_boundary_conditions, v=v_boundary_conditions, T=T_boundary_conditions))

    set!(model, T = (λ, φ, z) -> 10 * exp(z / 1000))

    return model
end

In [ ]:
year = 365days

function run_gyres(grid, coriolis, name; stop_time=6year, save_interval=10days)
    model = build_model(grid, coriolis)
    simulation = Simulation(model; Δt, stop_time)

    wall_clock = Ref(time_ns())

    function progress(sim)
        u, v, w = sim.model.velocities
        elapsed = 1e-9 * (time_ns() - wall_clock[])
        @info @sprintf("%s, iter: %d, time: %s, max|u|: %.2f m/s, wall time: %s",
                       name, iteration(sim), prettytime(sim), maximum(abs, u), prettytime(elapsed))
        wall_clock[] = time_ns()
        return nothing
    end

    add_callback!(simulation, progress, IterationInterval(500))

    u, v, w = model.velocities
    U = Field(Integral(u, dims=3))
    ψ = Field(CumulativeIntegral(-U, dims=2))
    speed = Field(@at (Center, Center, Center) sqrt(u^2 + v^2))
    surface_speed = view(speed, :, :, grid.Nz)
    surface_temperature = view(model.tracers.T, :, :, grid.Nz)

    filename = "global_wind_driven_gyres_$name.jld2"

    simulation.output_writers[:surface] = JLD2Writer(model, (; ψ, surface_speed, surface_temperature); filename,
                                                     schedule = TimeInterval(save_interval),
                                                     array_type = Array{Float32},
                                                     overwrite_files = true)

    run!(simulation)

    return filename
end

rotation_rates = (Ω, Ω / 2)
coriolis_titles = Dict(Ω => "f = 2Ω sin φ", Ω / 2 => "f = Ω sin φ")
filenames = Dict(rotation_rate => run_gyres(grid, HydrostaticSphericalCoriolis(; rotation_rate, scheme=DualGridScheme(grid)), @sprintf("omega_%g", rotation_rate / Ω))
                 for rotation_rate in rotation_rates)

f_plane_filename = run_gyres(grid, FPlane(latitude=30, scheme=DualGridScheme(grid)), "f_plane")

In [ ]:
times = FieldTimeSeries(filenames[Ω], "ψ").times

function gyre_transport(ψ, box)
    inside = Field{Face, Face, Nothing}(ψ.grid, Bool)
    set!(inside, (λ, φ) -> (box.longitude[1] < mod(λ, 360) < box.longitude[2]) &
                           (box.latitude[1] < φ < box.latitude[2]))
    return maximum(ψ; condition=inside) - minimum(ψ; condition=inside)
end

gulf_stream = (longitude = (275, 310), latitude = (20, 42))
kuroshio = (longitude = (118, 160), latitude = (20, 42))

In [ ]:
Sv = 1e6 # m³ s⁻¹

fig = Figure(size=(900, 400))
axes = (gulf_stream = Axis(fig[1, 1], title="Gulf Stream", xlabel="Time [years]", ylabel="Transport [Sv]"),
        kuroshio = Axis(fig[1, 2], title="Kuroshio", xlabel="Time [years]"))

colors = Dict(zip(rotation_rates, Makie.wong_colors()))

for rotation_rate in rotation_rates
    streamfunctions = FieldTimeSeries(filenames[rotation_rate], "ψ")
    label = coriolis_titles[rotation_rate]
    color = colors[rotation_rate]

    for (name, box) in ((:gulf_stream, gulf_stream), (:kuroshio, kuroshio))
        transport = [gyre_transport(streamfunctions[n], box) for n in 1:length(times)]
        lines!(axes[name], times / year, transport / Sv; label, color)
    end
end

axislegend(axes.gulf_stream, position=:lt)

In [ ]:
height_cpu = on_architecture(CPU(), height)
land = interior(height_cpu, :, :, 1) .≥ 0

distance_to_north_america = Field{Center, Center, Nothing}(underlying_grid)
set!(distance_to_north_america, (λ, φ) -> (mod(λ, 360) - 260)^2 + (φ - 40)^2)
north_america = argmin(distance_to_north_america)

function streamfunction_map!(fig, row, filename; title, colorrange)
    streamfunctions = FieldTimeSeries(filename, "ψ")
    ψ_end = streamfunctions[end]
    reference = ψ_end[north_america]
    streamfunction = Field((ψ_end - reference) / Sv)
    ax = map_axis(fig[row, 1]; title)
    qm = quadmesh!(ax, streamfunction; colormap=:balance, colorrange, nan_color=:gray)
    Colorbar(fig[row, 2], qm, label="Streamfunction [Sv]")
    return nothing
end

experiments = ((filenames[Ω], coriolis_titles[Ω], (-100, 100)),
               (filenames[Ω / 2], coriolis_titles[Ω / 2], (-100, 100)),
               (f_plane_filename, "f = 2Ω sin 30°", (-1000, 1000)))

fig = Figure(size=(900, 1050))

for (row, (filename, title, colorrange)) in enumerate(experiments)
    streamfunction_map!(fig, row, filename; title, colorrange)
end

In [ ]:
gulf_stream_view = (limits = ((260, 320), (15, 55)), xticks = (270:15:315, ["90°W", "75°W", "60°W", "45°W"]))
kuroshio_view = (limits = ((115, 175), (15, 55)), xticks = (120:15:165, ["120°E", "135°E", "150°E", "165°E"]))

tripolar_grid = TripolarGrid(CPU(), Float64; size=(Nx, Ny, 1), z=(0, 1)) ## the regridder needs Float64 coordinates
restoring_profile = Field{Center, Center, Nothing}(tripolar_grid)
set!(restoring_profile, (λ, φ) -> restoring_temperature(φ))

latitude_longitude_grid((longitude, latitude)) =
    LatitudeLongitudeGrid(CPU(), Float64; longitude, latitude, topology=(Bounded, Bounded, Flat),
                          size=(2 * (longitude[2] - longitude[1]), 2 * (latitude[2] - latitude[1])))

regions = ("Gulf Stream" => gulf_stream_view, "Kuroshio" => kuroshio_view)
regional_grids = [latitude_longitude_grid(zoom.limits) for (_, zoom) in regions]
regridders = [ConservativeRegridding.Regridder(grid, tripolar_grid) for grid in regional_grids]

function regrid_temperature_anomaly(temperatures)
    anomaly = Field{Center, Center, Nothing}(tripolar_grid)
    regional_anomalies = [FieldTimeSeries{Center, Center, Nothing}(grid, temperatures.times) for grid in regional_grids]

    for n in eachindex(temperatures.times)
        set!(anomaly, ifelse.(land, NaN, interior(temperatures[n], :, :, 1) .- interior(restoring_profile, :, :, 1)))
        for (anomalies, regridder) in zip(regional_anomalies, regridders)
            ConservativeRegridding.regrid!(anomalies[n], regridder, anomaly)
        end
    end

    return regional_anomalies
end

n = Observable(1)

fig = Figure(size=(1800, 1000))
Label(fig[1, 1:5], @lift(@sprintf("After %.1f years", times[$n] / year)); fontsize=22, tellwidth=false)

for (row, (filename, title, _)) in enumerate(experiments)
    speeds = FieldTimeSeries(filename, "surface_speed")
    speed = @lift speeds[$n]

    ax = map_axis(fig[row + 1, 1]; title="Surface speed, " * title)
    qm = quadmesh!(ax, speed; colormap=:magma, colorrange=(0, 0.3), nan_color=:gray)
    row == 1 && Colorbar(fig[2:4, 2], qm, label="Speed [m s⁻¹]")

    regional_anomalies = regrid_temperature_anomaly(FieldTimeSeries(filename, "surface_temperature"))

    for (column, ((region, zoom), anomalies)) in enumerate(zip(regions, regional_anomalies))
        ax = map_axis(fig[row + 1, column + 2]; title="T − T*, $region, " * title, zoom...)
        hm = heatmap!(ax, @lift(anomalies[$n]); colormap=:balance, colorrange=(-2, 2), nan_color=:gray)
        row == 1 && column == 2 && Colorbar(fig[2:4, 5], hm, label="T − T* [°C]")
    end
end

CairoMakie.record(fig, "global_wind_driven_gyres.mp4", 1:2:length(times), framerate=12) do frame
    n[] = frame
end